In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
from ingest import load_faq_data

documents = load_faq_data()

In [3]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [4]:
from tqdm.auto import tqdm

In [5]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [6]:
import numpy as np
X = np.array(vectors)

In [7]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [8]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [9]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [10]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

#Building RAG
using RAGBase from rag_helper.py


In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [12]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [13]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [14]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [15]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [16]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [17]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)